# Shift-click callback demo

This notebook demonstrates `ViewerConfig(on_shift_click=...)` and the optional `overlay_message`.
Shift-click inside the image to add/remove markers at continuous (fractional) data-space coordinates.
Clicks within 2 px of an existing marker remove that marker.

In [1]:
import numpy as np
import ipywidgets as widgets
import pyviewarr

In [2]:
# Build a simple smooth image so subpixel coordinates are easy to see.
ny, nx = 300, 420
y, x = np.mgrid[0:ny, 0:nx]
img = (
    np.exp(-(((x - 130.0) / 30.0) ** 2 + ((y - 110.0) / 25.0) ** 2))
    + 0.6 * np.exp(-(((x - 300.0) / 40.0) ** 2 + ((y - 190.0) / 35.0) ** 2))
).astype(np.float32)

In [3]:
import ipywidgets as widgets

output = widgets.Output()
display(output)

marked_points: list[tuple[float, float]] = []
widget = None

def on_shift_click(x: float, y: float) -> None:
    global marked_points, widget

    for i, (px, py) in enumerate(marked_points):
        if ((px - x) ** 2 + (py - y) ** 2) ** 0.5 < 2.0:
            removed = marked_points.pop(i)
            if widget is not None:
                widget.markers = list(marked_points)
            with output:
                print(f"Removed marker at x={removed[0]:.3f}, y={removed[1]:.3f}")
            return

    marked_points.append((x, y))
    if widget is not None:
        widget.markers = list(marked_points)
    with output:
        print(f"Added marker at x={x:.3f}, y={y:.3f} (count={len(marked_points)})")

cfg = pyviewarr.ViewerConfig(
    on_shift_click=on_shift_click,
    overlay_message="Shift-click to add/remove markers (<2 px)",
)
widget = pyviewarr.viewarr(img, viewer_config=cfg)
widget.markers = list(marked_points)
widget


Output()

In [7]:
marked_points

[(152.75198364257812, 141.19198608398438)]

In [8]:
widget.markers

[(152.75198364257812, 141.19198608398438)]